In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold

# Load Dataset

In [ ]:
# Load train and test data 
train = pd.read_csv("./data/train.csv")
test  = pd.read_csv("./data/test.csv")

# Re-parse kolom "Date"
train["Date"] = pd.to_datetime(train["Date"])
test["Date"]  = pd.to_datetime(test["Date"])

print(f"\ntrain: {train.shape}")
print(f"test : {test.shape}")
train.head()


train: (1866532, 10)
test : (465999, 10)


,Unnamed: 0,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,Target
0,0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,0.000730
1,1,20170104_7412,2017-01-04,7412,719.0,725.0,719.0,721.0,201400,0.000000
2,2,20170104_7408,2017-01-04,7408,2459.0,2518.0,2447.0,2500.0,110900,0.004421
3,3,20170104_7315,2017-01-04,7315,465.0,494.0,465.0,493.0,41100,-0.004032
4,4,20170104_7313,2017-01-04,7313,3055.0,3150.0,3045.0,3135.0,248600,-0.009693


In [7]:
#split data into train and validation sets
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
    print(f"Fold {fold}: train={len(train_idx)} rows, val={len(val_idx)} rows")

# ambil fold terakhir sebagai train/validation final
train_idx, val_idx = list(tscv.split(train))[-1]

train_sub = train.iloc[train_idx].reset_index(drop=True)
val_sub   = train.iloc[val_idx].reset_index(drop=True)

print(f"\nTrain final : {train_sub.shape}")
print(f"Validation  : {val_sub.shape}")

Fold 0: train=311092 rows, val=311088 rows
Fold 1: train=622180 rows, val=311088 rows
Fold 2: train=933268 rows, val=311088 rows
Fold 3: train=1244356 rows, val=311088 rows
Fold 4: train=1555444 rows, val=311088 rows

Train final : (1555444, 10)
Validation  : (311088, 10)


# Preprocessing

In [8]:
# drop missing values
train_sub = train_sub.dropna().reset_index(drop=True)
val_sub   = val_sub.dropna().reset_index(drop=True)
test      = test.dropna().reset_index(drop=True)

In [9]:
# Outlier Handling (Data Mentah) — Pembersihan Nilai Ekstrem & Error Input
def clean_raw_outliers(df, name="data"):
    """
    Membersihkan data mentah dari nilai ekstrem dan error input:
    1. Harga non-positif (harga <= 0 atau bernilai negatif) pada Open, High, Low, Close.
    2. Volume negatif (Volume < 0).
    3. Inkonsistensi struktural OHLC akibat typo/kesalahan input (High < Low, High < Open/Close, Low > Open/Close).
    4. Anomali lonjakan harga intraday ekstrem akibat kesalahan ketik desimal (High / Low > 2.5).
    """
    before_len = len(df)
    
    # 1. Validasi nilai positif untuk harga dan non-negatif untuk volume
    price_cols = ["Open", "High", "Low", "Close"]
    valid_price = (df[price_cols] > 0).all(axis=1)
    valid_volume = df["Volume"] >= 0
    
    # 2. Validasi konsistensi relasi OHLC (mencegah typo/error pencatatan)
    valid_ohlc = (
        (df["High"] >= df["Low"]) &
        (df["High"] >= df["Open"]) &
        (df["High"] >= df["Close"]) &
        (df["Low"] <= df["Open"]) &
        (df["Low"] <= df["Close"])
    )
    
    # 3. Validasi batas rasio harian ekstrem (High / Low) untuk mendeteksi typo nilai
    # Pada bursa saham (JPX), pergerakan harian dibatasi batas auto-rejection (~15-30%)
    price_ratio = df["High"] / df["Low"]
    valid_ratio = price_ratio <= 2.5
    
    # Gabungkan seluruh filter validitas
    valid_mask = valid_price & valid_volume & valid_ohlc & valid_ratio
    anomalies_count = int((~valid_mask).sum())
    
    cleaned_df = df.loc[valid_mask].reset_index(drop=True)
    
    print(f"[{name}] Pembersihan Outlier Mentah & Input Error:")
    print(f"  - Total baris sebelum   : {before_len:,}")
    print(f"  - Anomali/error dibuang : {anomalies_count:,} baris")
    print(f"  - Baris valid tersisa   : {len(cleaned_df):,}")
    
    return cleaned_df

# Terapkan pembersihan pada data mentah sebelum feature engineering
train_sub = clean_raw_outliers(train_sub, "train_sub")
val_sub   = clean_raw_outliers(val_sub, "val_sub")
test      = clean_raw_outliers(test, "test")


[train_sub] Pembersihan Outlier Mentah & Input Error:
  - Total baris sebelum   : 1,551,801
  - Anomali/error dibuang : 0 baris
  - Baris valid tersisa   : 1,551,801
[val_sub] Pembersihan Outlier Mentah & Input Error:
  - Total baris sebelum   : 308,228
  - Anomali/error dibuang : 0 baris
  - Baris valid tersisa   : 308,228
[test] Pembersihan Outlier Mentah & Input Error:
  - Total baris sebelum   : 464,894
  - Anomali/error dibuang : 0 baris
  - Baris valid tersisa   : 464,894


# Feature Engineering

In [10]:
train_sub = train_sub.copy()
val_sub   = val_sub.copy()
test      = test.copy()

train_sub["_split"] = "train_sub"
val_sub["_split"]   = "val_sub"
test["_split"]      = "test"
full = pd.concat([train_sub, val_sub, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)

g = full.groupby("SecuritiesCode", sort=False)

# 1. Fitur Temporal
full["year"]       = full["Date"].dt.year
full["month"]      = full["Date"].dt.month
full["day"]        = full["Date"].dt.day
full["dayofweek"]  = full["Date"].dt.dayofweek
full["quarter"]    = full["Date"].dt.quarter
full["dayofyear"]  = full["Date"].dt.dayofyear
full["weekofyear"] = full["Date"].dt.isocalendar().week.astype(int)

# 2. Fitur Return (N-day)
full["ret_1d"]  = g["Close"].pct_change()
full["ret_5d"]  = g["Close"].pct_change(5)
full["ret_20d"] = g["Close"].pct_change(20)

# 3. Fitur Lag (explicit shift)
full["ret_1d_lag1"] = g["ret_1d"].shift(1)
full["ret_1d_lag2"] = g["ret_1d"].shift(2)
full["ret_1d_lag3"] = g["ret_1d"].shift(3)

# 4. Fitur Rolling Window (agregasi statistik)
full["vol_5d"]  = g["ret_1d"].transform(lambda s: s.rolling(5,  min_periods=2).std())
full["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())

vol_ma20             = g["Volume"].transform(lambda s: s.rolling(20, min_periods=1).mean())
full["volume_ratio"] = full["Volume"] / vol_ma20
del vol_ma20

# 5. Fitur Bentuk Intraday
full["intraday_range"] = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]     = (full["Close"] - full["Open"]) / full["Open"]
full["high_close"]     = (full["High"] - full["Close"]) / full["Close"]
full["low_close"]      = (full["Close"] - full["Low"]) / full["Close"]

# 6. Fitur Volume
full["log_volume"] = np.log1p(full["Volume"])

# 7. Fitur Cross-sectional Rank
full["ret_rank"]    = full.groupby("Date")["ret_1d"].rank(pct=True)
full["volume_rank"] = full.groupby("Date")["Volume"].rank(pct=True)

# pisahkan kembali
train_sub = full[full["_split"] == "train_sub"].drop(columns="_split").reset_index(drop=True)
val_sub   = full[full["_split"] == "val_sub"].drop(columns="_split").reset_index(drop=True)
test      = full[full["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

print("Fitur ditambahkan: Temporal, Return, Lag, Rolling Window, Intraday, Volume, Rank.")
print(f"train_sub: {train_sub.shape}, val_sub: {val_sub.shape}, test: {test.shape}")

Fitur ditambahkan: Temporal, Return, Lag, Rolling Window, Intraday, Volume, Rank.
train_sub: (1551801, 33), val_sub: (308228, 33), test: (464894, 33)


In [11]:
#outlier handling :capping
from math import isnan

def handle_outliers(df, column, method="iqr", action="cap", bounds=None):
    if column not in df.columns:
        raise KeyError(f"Column tidak ditemukan: {column}")
    if not pd.api.types.is_numeric_dtype(df[column]):
        raise TypeError(f"Column harus numerik: {column}")
    if method not in {"iqr", "zscore"}:
        raise ValueError("method harus 'iqr' atau 'zscore'")
    if action not in {"remove", "cap"}:
        raise ValueError("action harus 'remove' atau 'cap'")

    result = df.copy()
    values = result[column]

    if bounds is None:
        if method == "iqr":
            q1, q3 = values.quantile([0.25, 0.75])
            spread = q3 - q1
            lower = q1 - 1.5 * spread
            upper = q3 + 1.5 * spread
        else:
            mean = values.mean()
            std = values.std()
            lower = mean - 3 * std
            upper = mean + 3 * std
    else:
        lower, upper = bounds

    outlier_mask = values.notna() & ((values < lower) | (values > upper))
    before_rows = len(result)
    outlier_count = int(outlier_mask.sum())

    if action == "remove":
        result = result.loc[~outlier_mask].reset_index(drop=True)
        changed = outlier_count
        print(f"[{column}] {method} removal: {outlier_count} baris dihapus")
    else:
        result[column] = values.clip(lower=lower, upper=upper)
        changed = int((result[column] != values).fillna(False).sum())
        print(f"[{column}] {method} capping: {changed} nilai dibatasi")

    print(f"  Sebelum: {before_rows:,} baris | Sesudah: {len(result):,} baris")
    print(f"  Batas: [{lower:.6g}, {upper:.6g}] | Outlier terdeteksi: {outlier_count:,}")
    return result, (lower, upper)


# --- Terapkan ke SEMUA kolom skewed: train_sub -> hitung bounds, val_sub & test -> pakai bounds train_sub ---
CAP_COLS = [
    "ret_1d", "ret_5d", "ret_20d",
    "ret_1d_lag1", "ret_1d_lag2", "ret_1d_lag3",
    "vol_5d", "vol_20d",
    "volume_ratio",
]

cap_bounds = {}
for col in CAP_COLS:
    train_sub, bounds = handle_outliers(train_sub, column=col, method="iqr", action="cap")
    cap_bounds[col] = bounds

for col in CAP_COLS:
    val_sub, _ = handle_outliers(val_sub, column=col, method="iqr", action="cap", bounds=cap_bounds[col])
    test, _    = handle_outliers(test, column=col, method="iqr", action="cap", bounds=cap_bounds[col])

print("\nSemua kolom skewed sudah di-cap (bounds dari train_sub):")
for col, (lo, hi) in cap_bounds.items():
    print(f"  {col:15s}: [{lo:.6g}, {hi:.6g}]")

[ret_1d] iqr capping: 109350 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403926, 0.0402078] | Outlier terdeteksi: 107,376
[ret_5d] iqr capping: 117121 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0963203, 0.0970418] | Outlier terdeteksi: 107,251
[ret_20d] iqr capping: 130836 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.208932, 0.211786] | Outlier terdeteksi: 91,356
[ret_1d_lag1] iqr capping: 111129 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403739, 0.0401712] | Outlier terdeteksi: 107,181
[ret_1d_lag2] iqr capping: 112987 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403688, 0.0401496] | Outlier terdeteksi: 107,065
[ret_1d_lag3] iqr capping: 114797 nilai dibatasi
  Sebelum: 1,551,801 baris | Sesudah: 1,551,801 baris
  Batas: [-0.0403737, 0.0401183] | Outlier terdeteksi: 106,901
[vol_5d] iqr capping:

In [12]:
#security code target encoding
mean_target = train_sub.groupby("SecuritiesCode")["Target"].mean()
global_mean = train_sub["Target"].mean()

train_sub["SecuritiesCode_te"] = train_sub["SecuritiesCode"].map(mean_target)
val_sub["SecuritiesCode_te"]   = val_sub["SecuritiesCode"].map(mean_target).fillna(global_mean)
test["SecuritiesCode_te"]      = test["SecuritiesCode"].map(mean_target).fillna(global_mean)

print(f"SecuritiesCode_te dibuat. mean (train_sub)={train_sub['SecuritiesCode_te'].mean():.6f}")
print(f"SecuritiesCode_te dibuat. mean (val_sub)={val_sub['SecuritiesCode_te'].mean():.6f}")
print(f"SecuritiesCode_te dibuat. mean (test)={test['SecuritiesCode_te'].mean():.6f}")

SecuritiesCode_te dibuat. mean (train_sub)=0.000204
SecuritiesCode_te dibuat. mean (val_sub)=0.000267
SecuritiesCode_te dibuat. mean (test)=0.000266


In [13]:
# Finalisasi dataset untuk train_sub, val_sub, dan test
# Kolom OHLCV tidak digunakan sebagai fitur model setelah feature engineering.
feature_drop = ["Open", "High", "Low", "Close", "Volume"]
identifier_drop = ["Unnamed: 0", "RowId", "SecuritiesCode", "Date"]

train_final = train_sub.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)
val_final   = val_sub.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)
test_final  = test.drop(columns=feature_drop, errors="ignore").dropna().reset_index(drop=True)

train_X = train_final.drop(columns="Target", errors="ignore")
train_y = train_final["Target"]
val_X   = val_final.drop(columns="Target", errors="ignore")
val_y   = val_final["Target"]
test_X  = test_final.drop(columns="Target", errors="ignore")
test_y  = test_final["Target"]                      

train_X = train_X.drop(columns=identifier_drop, errors="ignore")
val_X   = val_X.drop(columns=identifier_drop, errors="ignore")
test_X  = test_X.drop(columns=identifier_drop, errors="ignore")

print(f"Fitur final ({len(train_X.columns)} kolom): {list(train_X.columns)}")
print(f"train_sub: {train_X.shape}, target: {train_y.shape}")
print(f"val_sub  : {val_X.shape}, target: {val_y.shape}")
print(f"test     : {test_X.shape}, target: {test_y.shape}")   

Fitur final (24 kolom): ['year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear', 'ret_1d', 'ret_5d', 'ret_20d', 'ret_1d_lag1', 'ret_1d_lag2', 'ret_1d_lag3', 'vol_5d', 'vol_20d', 'volume_ratio', 'intraday_range', 'close_open', 'high_close', 'low_close', 'log_volume', 'ret_rank', 'volume_rank', 'SecuritiesCode_te']
train_sub: (1512321, 24), target: (1512321,)
val_sub  : (307872, 24), target: (307872,)
test     : (464730, 24), target: (464730,)


In [14]:
# Feature scaling
from sklearn.preprocessing import StandardScaler, MinMaxScaler

FEATURE_COLS = train_X.columns.tolist()
TARGET_COL = "Target"

X_train = train_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_train = train_y.to_numpy(dtype=np.float64).reshape(-1, 1)
X_val   = val_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_val   = val_y.to_numpy(dtype=np.float64).reshape(-1, 1)
X_test  = test_X[FEATURE_COLS].to_numpy(dtype=np.float64)
y_test  = test_y.to_numpy(dtype=np.float64).reshape(-1, 1)      

# Pipeline Machine Learning
scaler_X_svr = StandardScaler()
scaler_y_svr = StandardScaler()
X_train_svr = scaler_X_svr.fit_transform(X_train)
X_val_svr   = scaler_X_svr.transform(X_val)
X_test_svr  = scaler_X_svr.transform(X_test)
y_train_svr = scaler_y_svr.fit_transform(y_train).ravel()
y_val_svr   = scaler_y_svr.transform(y_val).ravel()
y_test_svr  = scaler_y_svr.transform(y_test).ravel()             

# Pipeline Deep Learning
scaler_X_lstm = MinMaxScaler()
scaler_y_lstm = MinMaxScaler()
X_train_lstm = scaler_X_lstm.fit_transform(X_train)
X_val_lstm   = scaler_X_lstm.transform(X_val)
X_test_lstm  = scaler_X_lstm.transform(X_test)
y_train_lstm = scaler_y_lstm.fit_transform(y_train).ravel()
y_val_lstm   = scaler_y_lstm.transform(y_val).ravel()
y_test_lstm  = scaler_y_lstm.transform(y_test).ravel()           

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

Jumlah fitur: 24
X_train: (1512321, 24) | X_val: (307872, 24) | X_test: (464730, 24)


# Modelling

In [15]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
# Build LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

n_features = X_train_lstm.shape[1]

X_train_lstm_3d = X_train_lstm.reshape(X_train_lstm.shape[0], 1, n_features)
X_val_lstm_3d   = X_val_lstm.reshape(X_val_lstm.shape[0], 1, n_features)
X_test_lstm_3d  = X_test_lstm.reshape(X_test_lstm.shape[0], 1, n_features)

lstm_model = Sequential([
    Input(shape=(1, n_features)),
    LSTM(50, activation="relu"),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mean_squared_error")
lstm_model.summary()

print(f"X_train_lstm: {X_train_lstm_3d.shape}")
print(f"X_val_lstm  : {X_val_lstm_3d.shape}")
print(f"X_test_lstm : {X_test_lstm_3d.shape}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        15,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,051 (58.79 KB)

 Trainable params: 15,051 (58.79 KB)

 Non-trainable params: 0 (0.00 B)

X_train_lstm: (1512321, 1, 24)
X_val_lstm  : (307872, 1, 24)
X_test_lstm : (464730, 1, 24)


In [17]:
# training model
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print(f"Training LSTM pada {len(X_train_lstm_3d):,} sample train "
      f"dan {len(X_val_lstm_3d):,} sample validation (maks 100 epoch) ...")

history = lstm_model.fit(
    X_train_lstm_3d, y_train_lstm,              
    validation_data=(X_val_lstm_3d, y_val_lstm),
    batch_size=16,
    epochs=100,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False,
)

Training LSTM pada 1,512,321 sample train dan 307,872 sample validation (maks 100 epoch) ...
Epoch 1/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 203s 2ms/step - loss: 5.1920e-04 - val_loss: 4.8358e-04
Epoch 2/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 170s 2ms/step - loss: 3.8376e-04 - val_loss: 4.8534e-04
Epoch 3/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 199s 2ms/step - loss: 3.7857e-04 - val_loss: 4.9120e-04
Epoch 4/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 262s 3ms/step - loss: 3.7651e-04 - val_loss: 4.8259e-04
Epoch 5/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 243s 3ms/step - loss: 3.7483e-04 - val_loss: 4.8890e-04
Epoch 6/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 254s 3ms/step - loss: 3.7364e-04 - val_loss: 4.8859e-04
Epoch 7/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 265s 3ms/step - loss: 3.7267e-04 - val_loss: 4.8520e-04
Epoch 8/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 307s 3ms/step - loss: 3.7132e-04 - val_loss: 4.8568e-04
Epoch 9/100
94521/94521 ━━━━━━━━━━━━━━━━━━━━ 237s 3ms/step - loss: 3.7025e-04 - val_loss: 4.8893e-0

In [18]:
# evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Generate predictions on the test set in the scaled target space.
y_pred_lstm_scaled = lstm_model.predict(X_test_lstm_3d, verbose=0).ravel()
y_true_lstm_scaled = y_test_lstm.ravel()

# Inverse transform back to the original target scale for meaningful metrics.
y_pred_lstm = scaler_y_lstm.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).ravel()
y_true_lstm = scaler_y_lstm.inverse_transform(y_true_lstm_scaled.reshape(-1, 1)).ravel()

mse_lstm = mean_squared_error(y_true_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = mean_absolute_error(y_true_lstm, y_pred_lstm)

print(f"LSTM -> MSE: {mse_lstm:.8f} | RMSE: {rmse_lstm:.8f} | MAE: {mae_lstm:.8f}")

LSTM -> MSE: 0.00059868 | RMSE: 0.02446790 | MAE: 0.01734428


# Hyperparameter Tuning

In [19]:
%pip install optuna

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import optuna
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, Callback
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed()

c:\Users\grace\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
n_features = X_train_lstm_3d.shape[2]

def build_lstm(params):
    model = Sequential()
    model.add(Input(shape=(1, n_features)))

    n_layers = params["n_layers"]
    for i in range(n_layers):
        model.add(LSTM(
            params[f"units_l{i}"],
            activation=params["activation"],
            return_sequences=(i < n_layers - 1),  # layer terakhir tidak return sequences
        ))
        model.add(Dropout(params["dropout"]))

    model.add(Dense(1))
    model.compile(
        optimizer=Adam(learning_rate=params["lr"]),
        loss="mean_squared_error",
    )
    return model


class OptunaPruningCallback(Callback):
    """Laporkan val_loss tiap epoch ke Optuna & hentikan trial yang tidak menjanjikan."""
    def __init__(self, trial):
        super().__init__()
        self.trial = trial

    def on_epoch_end(self, epoch, logs=None):
        val_loss = logs.get("val_loss")
        self.trial.report(val_loss, step=epoch)
        if self.trial.should_prune():
            self.model.stop_training = True
            self.trial.set_user_attr("pruned_at_epoch", epoch)
            raise optuna.TrialPruned()

def objective(trial):
    tf.keras.backend.clear_session()   # bebaskan memori antar trial
    set_seed()

    n_layers = trial.suggest_int("n_layers", 1, 3)
    params = {
        "n_layers":   n_layers,
        "activation": trial.suggest_categorical("activation", ["tanh", "relu"]),
        "dropout":    trial.suggest_float("dropout", 0.0, 0.5, step=0.1),
        "lr":         trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256]),
    }
    for i in range(n_layers):
        params[f"units_l{i}"] = trial.suggest_categorical(f"units_l{i}", [16, 32, 64, 128])

    model = build_lstm(params)

    history = model.fit(
        X_train_lstm_3d, y_train_lstm,
        validation_data=(X_val_lstm_3d, y_val_lstm),
        batch_size=params["batch_size"],
        epochs=50,
        shuffle=False,
        verbose=0,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
            OptunaPruningCallback(trial),
        ],
    )

    trial.set_user_attr("best_epoch", int(np.argmin(history.history["val_loss"])) + 1)
    return float(min(history.history["val_loss"]))

In [22]:
optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(
    study_name="lstm_tuning",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
)

study.optimize(
    objective,
    n_trials=30,          # naikkan jika waktu cukup
    # timeout=3600,       # opsional: batasi total waktu (detik)
    show_progress_bar=True,
)

print("\n=== Hasil Tuning ===")
print(f"Best val_loss : {study.best_value:.8f}")
print("Best params   :")
for k, v in study.best_params.items():
    print(f"  {k:12s}: {v}")
print(f"Best epoch    : {study.best_trial.user_attrs.get('best_epoch')}")

[I 2026-09-28 09:12:49,324] A new study created in memory with name: lstm_tuning
  0%|          | 0/30 [00:00<?, ?it/s]

Best trial: 0. Best value: 0.000482147:   3%|▎         | 1/30 [24:16<11:43:59, 1456.53s/it]

[I 2026-09-28 09:37:05,852] Trial 0 finished with value: 0.0004821470065508038 and parameters: {'n_layers': 2, 'activation': 'tanh', 'dropout': 0.30000000000000004, 'lr': 2.9380279387035334e-05, 'batch_size': 128, 'units_l0': 64, 'units_l1': 128}. Best is trial 0 with value: 0.0004821470065508038.


Best trial: 1. Best value: 0.000482064:   7%|▋         | 2/30 [51:42<12:11:37, 1567.76s/it]

[I 2026-09-28 10:04:31,475] Trial 1 finished with value: 0.00048206408973783255 and parameters: {'n_layers': 2, 'activation': 'tanh', 'dropout': 0.30000000000000004, 'lr': 2.621087878265438e-05, 'batch_size': 256, 'units_l0': 64, 'units_l1': 128}. Best is trial 1 with value: 0.00048206408973783255.


...

[I 2026-09-28 13:32:27,997] Trial 27 pruned. 


Best trial: 17. Best value: 0.000475635:  97%|█████████▋| 29/30 [4:26:54<04:20, 260.03s/it]

[I 2026-09-28 13:39:43,630] Trial 28 finished with value: 0.0004792422987520695 and parameters: {'n_layers': 1, 'activation': 'relu', 'dropout': 0.4, 'lr': 0.0007531592884713922, 'batch_size': 256, 'units_l0': 64}. Best is trial 17 with value: 0.0004756352864205837.


Best trial: 17. Best value: 0.000475635: 100%|██████████| 30/30 [6:13:41<00:00, 747.38s/it] 

[I 2026-09-28 15:26:30,766] Trial 29 finished with value: 0.0004801145114470273 and parameters: {'n_layers': 1, 'activation': 'relu', 'dropout': 0.5, 'lr': 0.0003438143565586761, 'batch_size': 64, 'units_l0': 64}. Best is trial 17 with value: 0.0004756352864205837.

=== Hasil Tuning ===
Best val_loss : 0.00047564
Best params   :
  n_layers    : 2
  activation  : relu
  dropout     : 0.5
  lr          : 0.0005087550147469852
  batch_size  : 256
  units_l0    : 64
  units_l1    : 32
Best epoch    : 17


# Feature Selection